In [1]:
import numpy as np
from scipy.io import wavfile
from scipy.signal import get_window
import scipy.fftpack as fft
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [ ]:
AUDIO_PATH = './sample.wav'
sr, audio = wavfile.read(AUDIO_PATH)

print(sr)
print(audio)
print(audio.shape)

44100
[[  0   0]
 [  0   1]
 [ -1  -1]
 ...
 [-57 -57]
 [-57 -58]
 [-61 -60]]
(809508, 2)


C:\Users\prk\AppData\Local\Temp\ipykernel_18092\3091797834.py:2: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio = wavfile.read(AUDIO_PATH)


In [3]:
#Konfersi Stereo ke Mono
if audio.ndim > 1:
    audio = np.mean(audio, axis=1)

In [4]:
print(sr)
print(audio)
print(audio.shape)
print(audio.ndim)

44100
[  0.    0.5  -1.  ... -57.  -57.5 -60.5]
(809508,)
1


In [5]:
FFT_SIZE = 2048
HOP_SIZE = 15
pad_width = int(FFT_SIZE / 2)

#Normalization
print("Before normalization:", np.max(np.abs(audio)))
audio = audio / np.max(np.abs(audio))
print("After normalization:", np.max(np.abs(audio)))

Before normalization: 32768.0
After normalization: 1.0


In [6]:
#Frame Blocking
audio = np.pad(audio, pad_width, mode="reflect")
HOP_LENGTH = int(np.round(sr * HOP_SIZE / 1000))
FRAME_NUM = int(((len(audio) - FFT_SIZE) // HOP_LENGTH) + 1)

In [7]:
print("Hop Length:", HOP_LENGTH)
print("Frame Number:", FRAME_NUM)

Hop Length: 662
Frame Number: 1223


In [8]:
frames = np.zeros((FRAME_NUM, FFT_SIZE))

for i in range(FRAME_NUM):
    start = i * HOP_LENGTH
    end = start + FFT_SIZE
    frames[i] = audio[start:end]

In [9]:
print("Frames:", frames)

Frames: [[ 1.52587891e-05 -1.52587891e-05  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00 -1.52587891e-05]
 [ 0.00000000e+00  1.52587891e-05  0.00000000e+00 ...  0.00000000e+00
   1.52587891e-05  0.00000000e+00]
 [-1.52587891e-05 -1.52587891e-05  0.00000000e+00 ...  1.52587891e-05
  -1.52587891e-05  0.00000000e+00]
 ...
 [-7.47680664e-04 -5.03540039e-04 -4.73022461e-04 ...  8.39233398e-04
   9.00268555e-04  7.62939453e-04]
 [ 8.39233398e-04  5.79833984e-04  5.95092773e-04 ... -1.57165527e-03
  -1.51062012e-03 -1.73950195e-03]
 [ 1.22070312e-04  0.00000000e+00  3.05175781e-05 ...  1.00708008e-03
   1.31225586e-03  1.34277344e-03]]


In [10]:
#Windowing
window_type = "hann"
window = get_window(window_type, FFT_SIZE, fftbins=True)
audio_win = frames * window
print("Windowed Audio:", audio_win)

Windowed Audio: [[ 0.00000000e+00 -3.59053835e-11  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00 -3.59053835e-11]
 [ 0.00000000e+00  3.59053835e-11  0.00000000e+00 ...  0.00000000e+00
   1.43621196e-10  0.00000000e+00]
 [-0.00000000e+00 -3.59053835e-11  0.00000000e+00 ...  3.23146424e-10
  -1.43621196e-10  0.00000000e+00]
 ...
 [-0.00000000e+00 -1.18487765e-09 -4.45225708e-09 ...  1.77730533e-08
   8.47365056e-09  1.79526917e-09]
 [ 0.00000000e+00  1.36440457e-09  5.60122664e-09 ... -3.32840816e-08
  -1.42184984e-08 -4.09321372e-09]
 [ 0.00000000e+00  0.00000000e+00  2.87242392e-10 ...  2.13276640e-08
   1.23514229e-08  3.15967375e-09]]


In [11]:
#FFT
audio_winT = np.transpose(audio_win)
audio_fft = np.empty(
    (1 + FFT_SIZE // 2, audio_winT.shape[1]),
    dtype = np.complex64
)

for i in range(audio_fft.shape[1]):
    result = fft.fft(audio_winT[:, i])
    audio_fft[:, i] = result[:audio_fft.shape[0]]

audio_fft = np.transpose(audio_fft)

In [12]:
print("Audio FFT:", audio_fft)

Audio FFT: [[-1.6157106e-04-0.0000000e+00j  3.0967905e-04+3.1922554e-20j
  -4.7939498e-04+5.1284943e-20j ... -3.5632786e-04-6.5049225e-20j
   1.8408285e-04+9.8838157e-20j -2.0479815e-04-0.0000000e+00j]
 [ 1.4233899e-04-0.0000000e+00j -1.9459927e-04+8.3033236e-05j
   1.0129203e-04+1.2873835e-04j ... -2.3146005e-05-2.0495633e-04j
   7.2684095e-05+1.6882512e-04j -3.2179197e-04-0.0000000e+00j]
 [ 4.0165114e-04-0.0000000e+00j -1.8852919e-04+1.7924873e-04j
  -1.6916118e-04-1.5426207e-04j ... -2.5290169e-04+3.6013906e-05j
   3.4092763e-04-1.5115984e-04j -4.2336443e-04-0.0000000e+00j]
 ...
 [-3.6819603e-02-0.0000000e+00j  1.9541071e-01-7.3422782e-02j
  -3.2354212e-01-2.4210786e-02j ...  5.3050986e-04+4.0177131e-04j
  -4.7969559e-04-5.3890247e-04j  5.8824726e-04-0.0000000e+00j]
 [ 1.6503140e-01-0.0000000e+00j -1.4479044e-01+1.3368958e-01j
   3.1383939e-02-2.5650556e-03j ... -4.0442243e-04-2.5634575e-04j
  -3.1898959e-04-5.8966014e-04j  1.8043553e-03-0.0000000e+00j]
 [-7.9659499e-02-0.0000000e+0

In [ ]:
#Convert into power spectrum
audio_power = np.square(np.abs(audio_fft))

print(audio_power)

print(audio_power.shape)

[[2.6105209e-08 9.5901115e-08 2.2981955e-07 ... 1.2696954e-07
  3.3886494e-08 4.1942283e-08]
 [2.0260389e-08 4.4763393e-08 2.6833638e-08 ... 4.2542837e-08
  3.3784900e-08 1.0355007e-07]
 [1.6132364e-07 6.7673362e-08 5.2412290e-08 ... 6.5256259e-08
  1.3908095e-07 1.7923745e-07]
 ...
 [1.3556832e-03 4.3576248e-02 1.0526567e-01 ... 4.4286088e-07
  5.2052371e-07 3.4603485e-07]
 [2.7235365e-02 3.8837180e-02 9.9153107e-04 ... 2.2927065e-07
  4.4945347e-07 3.2556979e-06]
 [6.3456357e-03 1.4554833e-01 7.9397030e-02 ... 4.4780800e-06
  8.5917736e-06 1.3488065e-05]]
(1223, 1025)
